## Prerequisites

Before running this notebook, generate and save the robot mounting graph **once**:

```bash
python generate_mounting_graph.py
```

This is an interactive step — the Open3D window lets you crop unwanted surface regions. The result is saved to `data/mounting_graph.pkl` and loaded automatically when the notebook imports `config.params`.

# Import

In [ ]:
%load_ext autoreload
%autoreload 2

import random

from deap import base, creator, tools

import config.params as params
from config.params import Sensor, Gene, Individual, Population
import config.seeding as seeding
import config.sensors as sensors

# Initialization

In [ ]:
# Define the fitness to maximize
creator.create("FitnessMax", base.Fitness, weights=(1.0,))

# Define the Individual class as a list of Gene objects with an associated fitness.
creator.create("Individual", list, fitness=creator.FitnessMax)  # type: ignore

toolbox = base.Toolbox()

In [ ]:
import custom_toolbox.initialize.initialize as initialize

# Register the individual generator (oriented spawning: outward + slight downtilt).
toolbox.register("individual", initialize.create_individual, creator.Individual)    # type: ignore

# Fully random spawn generator, kept for A/B comparison (see "Compare spawn initialization").
toolbox.register("individual_random", initialize.create_individual, creator.Individual, oriented=False)  # type: ignore

# Register the population generator (creates a list of individuals).
toolbox.register("population", tools.initRepeat, list, toolbox.individual)  # type: ignore

# NOTE: Consider Demes
# A deme is a sub-population that is contained in a population.
# https://deap.readthedocs.io/en/master/tutorials/basic/part1.html#demes

# Create initial population.
population = initialize.create_seeded_population(
    creator.Individual, # type: ignore
    toolbox.individual, # type: ignore
    seeding.SEED_INDIVIDUALS,
    params.POPULATION_SIZE,
)

In [ ]:
import utils.visualization as visualization

visualization.visualize_population(population, max_display=10)

# Evaluation

In [ ]:
import custom_toolbox.evaluate.evaluate_fitness_raycast as evaluate

# Build the ray-cast coverage evaluator once (loads chassis mesh + raycasting scene).
evaluator = evaluate.CoverageEvaluator()

# Register the custom evaluation function (use the standard 'evaluate' name).
toolbox.register("evaluate", evaluator.evaluate_individual)

# Selection

In [ ]:
# Tournament selection with tournament size of 3.
toolbox.register("select", tools.selTournament, tournsize=3)

# Mutation

In [ ]:
import custom_toolbox.mutate.mutate as mutate

# Register the custom mutation function.
toolbox.register("mutate", mutate.mutate_sensor_layout)

# Mating

In [ ]:
import custom_toolbox.mate.mate as mate
import custom_toolbox.mate.mate_spatial as mate_spatial

# Register the custom crossover function.
# Spatial (front/back) crossover: recombine parents by where their sensors sit
# on the robot's length (x) axis, split at the median sensor x of the pair.
toolbox.register("mate", mate_spatial.cx_spatial_front_back)

# Previous index-based one-point operator (kept for A/B comparison):
# toolbox.register("mate", mate.cx_variable_length_bounded)

# Evolution

In [ ]:
import numpy as np
from deap import tools

# 1. Create a Statistics object that looks at the first objective values
stats = tools.Statistics(key=lambda ind: ind.fitness.values[0])

# 2. Register the metrics you want to track
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)

# 3. Create a Logbook to store the data nicely
logbook = tools.Logbook()
logbook.header = ["gen", "nevals"] + (stats.fields if stats else [])

In [ ]:
from utils.ga_run import (
    run_evolution,
    TOURNAMENT,                   # global elitism + tournament
    TOURNAMENT_PER_LENGTH_ELITE,  # per-length elitism + tournament
    LENGTH_NICHING,               # (mu+lambda) length niching
)

# Single-run strategy. One of:
#   TOURNAMENT, TOURNAMENT_PER_LENGTH_ELITE, LENGTH_NICHING
# (To compare strategies head to head, see "Compare selection strategies" below.)
SELECTION_STRATEGY = LENGTH_NICHING

result = run_evolution(
    population,
    toolbox,
    strategy=SELECTION_STRATEGY,
    seed=42,
)

# Unpack into the names the visualization cells below expect.
logbook = result.logbook
hof = result.hof
best_per_length = result.best_per_length
per_length_evolution = result.per_length_evolution
population[:] = result.population

print("Best overall layout found:", hof[0])
print("Number of sensors in best layout:", len(hof[0]))
print("Best overall fitness:", hof[0].fitness.values)

print("\n-- Best layout per sensor count --")
for length in best_per_length.lengths:
    best = best_per_length[length]
    print(f"  {length} sensor(s): fitness={best.fitness.values[0]:.4f}")


In [ ]:
import utils.visualization as visualization

visualization.visualize_evolution(logbook)

In [ ]:
visualization.visualize_evolution_per_length(per_length_evolution)

In [ ]:
visualization.visualize_length_distribution(per_length_evolution)

In [ ]:
visualization.visualize_coverage_maps(hof[0], evaluator=evaluator)

In [ ]:
visualization.visualize_best_layout(hof[0], evaluator=evaluator)

# Best Layout per Sensor Count

`best_per_length` holds the fittest individual found at each sensor count (1..`MAX_SENSORS_PER_INDIVIDUAL`). The comparison view shows the fitness/cost tradeoff across counts; the detail view renders each champion's coverage maps and 3D layout.

visualization.visualize_length_comparison(best_per_length, evaluator=evaluator)

visualization.visualize_bests_per_length(best_per_length, evaluator=evaluator)

# Compare selection strategies

Run the selected strategies from the **same initial population and seed**, so the only controlled difference is the selection scheme. By default this compares the two tournament variants:

- **Tournament + global elite** — carries the `ELITE_COUNT` best individuals *overall* (protects only the currently-dominant sensor count).
- **Tournament + per-length elite** — carries the `ELITE_COUNT` best of *each* sensor count, so every count keeps champions forward.

(Append `LENGTH_NICHING` to `STRATEGIES` to add the (mu+lambda) niching scheme as a third line.) The plots:

- **Fitness over generations** — does either scheme reach a better overall layout?
- **Population composition** — per-length elitism should slow the collapse toward a single count vs global elitism.
- **Best per sensor count** — which scheme finds the better layout at each count?
- **Per-count evolution** — small multiples; a broken line means that count went extinct under that scheme.

In [ ]:
import config.seeding as seeding
import custom_toolbox.initialize.initialize as initialize
from utils.ga_run import (
    run_evolution,
    TOURNAMENT,                   # global elitism + tournament
    TOURNAMENT_PER_LENGTH_ELITE,  # per-length elitism + tournament
    LENGTH_NICHING,               # (mu+lambda) length niching
)
import utils.comparison as comparison

# Strategies to compare. Defaults to the two tournament variants (global vs
# per-length elitism); append LENGTH_NICHING to bring in niching as a third line.
STRATEGIES = [TOURNAMENT, LENGTH_NICHING]

# Build ONE fresh starting population and run every strategy from it with the
# same seed, so the only difference between the runs is the selection scheme.
# (Tip: lower params.NGEN / params.POPULATION_SIZE for a quick A/B; each run is
# a full evolution. The shared evaluator cache makes later runs cheaper.)
initial_population = initialize.create_seeded_population(
    creator.Individual,  # type: ignore
    toolbox.individual,  # type: ignore
    seeding.SEED_INDIVIDUALS,
    params.POPULATION_SIZE,
)

COMPARE_SEED = 11

results = {}
for strat in STRATEGIES:
    res = run_evolution(
        initial_population,
        toolbox,
        strategy=strat,
        seed=COMPARE_SEED,
    )
    results[res.label] = res

In [ ]:
# Overall best (solid) and average (dashed) fitness per strategy.
comparison.compare_evolution(results)

In [ ]:
# Population composition over generations, one panel per strategy. This is the
# headline contrast: length niching holds every sensor-count band open, while
# tournament + elitism tends to let one count dominate the population.
comparison.compare_length_distribution(results, normalize=True)

In [ ]:
# Best layout found at each sensor count, per strategy (table + grouped bars).
# Passing the evaluator adds a coverage % column (recomputes coverage per champion).
comparison.compare_best_per_length(results, evaluator=evaluator)

In [ ]:
# Per sensor count, both strategies overlaid. A count whose line breaks (e.g. a
# late gap) went extinct under that strategy; niching should keep every count's
# line continuous and improving.
comparison.compare_per_length_evolution(results, metric="max")

# Compare mating strategies

Hold the algorithm fixed at the original **Tournament + global elite** selection and vary **only the crossover operator**, to isolate its effect:

- **Spatial front/back** (`mate_spatial.cx_spatial_front_back`) — recombines parents by *where their sensors sit* on the robot's length axis (split at the pair's median sensor x).
- **One-point bounded** (`mate.cx_variable_length_bounded`) — index-based variable-length one-point splice.

Both run from the **same initial population and seed**, at full scale, via the `mate=` override on `run_evolution`, so the only controlled difference is the crossover operator. The winner is judged on **best overall fitness** (hall of fame).

> ⚠️ **Single seed (n=1).** As the selection-strategy verification showed, one seed can land on the *opposite* of the true average. Treat this as indicative only; for a defensible verdict, repeat across several seeds and compare the mean ± spread.

In [ ]:
import config.seeding as seeding
import custom_toolbox.initialize.initialize as initialize
import custom_toolbox.mate.mate as mate
import custom_toolbox.mate.mate_spatial as mate_spatial
from utils.ga_run import run_evolution, TOURNAMENT, TOURNAMENT_PER_LENGTH_ELITE
import utils.comparison as comparison

# Vary ONLY the crossover operator; everything else (selection = TOURNAMENT
# global elite, mutation, seed, initial population, scale) is held fixed.
MATE_OPERATORS = {
    "Spatial front/back": mate_spatial.cx_spatial_front_back,
    "One-point bounded": mate.cx_variable_length_bounded,
}

# One fresh starting population, reused (deep-copied inside) by both operators.
mate_initial_population = initialize.create_seeded_population(
    creator.Individual,  # type: ignore
    toolbox.individual,  # type: ignore
    seeding.SEED_INDIVIDUALS,
    params.POPULATION_SIZE,
)

MATE_COMPARE_SEED = 23

mate_results = {}
for op_label, op in MATE_OPERATORS.items():
    res = run_evolution(
        mate_initial_population,
        toolbox,
        strategy=TOURNAMENT_PER_LENGTH_ELITE,  # original global-elite tournament, fixed
        mate=op,  # the only thing that varies
        label=op_label,
        seed=MATE_COMPARE_SEED,
    )
    mate_results[res.label] = res

In [ ]:
# Overall best (solid) and average (dashed) fitness per crossover operator.
comparison.compare_evolution(mate_results)

In [ ]:
# Population composition over generations, one panel per operator. Shows whether
# spatial recombination keeps sensor-count bands open differently than one-point.
comparison.compare_length_distribution(mate_results, normalize=True)

In [ ]:
# Best layout found at each sensor count, per operator (table + grouped bars).
# Passing the evaluator adds a coverage % column (recomputes coverage per champion).
comparison.compare_best_per_length(mate_results, evaluator=evaluator)

In [ ]:
# Per sensor count, both operators overlaid (best fitness over generations).
comparison.compare_per_length_evolution(mate_results, metric="max")

# Compare mutation strategies

Hold the algorithm fixed at the original **Tournament + global elite** selection and crossover, and vary **only the mutation scheme**:

- **Uniform mutation** (`tiered_mutation=False`) — every individual gets the same perturbation (the baseline; MEDIUM tier = the operator's original constants).
- **Fitness-tiered mutation** (`tiered_mutation=True`) — each generation the population is split by global fitness rank into superior (10%, *exploit* — tiny perturbation), medium (40%, balance), and inferior (50%, *explore* — large perturbation + multi-hop moves).

Both run from the **same initial population and seed**, at full scale, so the only controlled difference is the mutation scheme. Judge on **best overall fitness** (hall of fame).

> ⚠️ **Single seed (n=1).** One seed can land on the opposite of the true average — treat as indicative; repeat across several seeds for a defensible verdict.

In [ ]:
import config.seeding as seeding
import custom_toolbox.initialize.initialize as initialize
from utils.ga_run import run_evolution, TOURNAMENT, TOURNAMENT_PER_LENGTH_ELITE
import utils.comparison as comparison

# Vary ONLY the mutation scheme; selection (TOURNAMENT global elite), crossover,
# seed, initial population and scale are all held fixed.
MUTATION_SCHEMES = {
    "Uniform mutation": False,
    "Fitness-tiered mutation": True,
}

# One fresh starting population, reused (deep-copied inside) by both schemes.
mut_initial_population = initialize.create_seeded_population(
    creator.Individual,  # type: ignore
    toolbox.individual,  # type: ignore
    seeding.SEED_INDIVIDUALS,
    params.POPULATION_SIZE,
)

MUT_COMPARE_SEED = 23

mut_results = {}
for label, tiered in MUTATION_SCHEMES.items():
    res = run_evolution(
        mut_initial_population,
        toolbox,
        strategy=TOURNAMENT,        # original global-elite tournament, fixed
        tiered_mutation=tiered,     # the only thing that varies
        label=label,
        seed=MUT_COMPARE_SEED,
    )
    mut_results[res.label] = res

In [ ]:
# Overall best (solid) and average (dashed) fitness per mutation scheme.
comparison.compare_evolution(mut_results)

In [ ]:
# Population composition over generations, one panel per scheme. Shows whether
# the inferior tier's stronger exploration keeps sensor-count bands open longer.
comparison.compare_length_distribution(mut_results, normalize=True)

In [ ]:
# Best layout per sensor count, per scheme (table + grouped bars).
comparison.compare_best_per_length(mut_results, evaluator=evaluator)

In [ ]:
# Per sensor count, both schemes overlaid (best fitness over generations).
comparison.compare_per_length_evolution(mut_results, metric="max")

# Compare spawn initialization

Hold the algorithm fixed at the original **Tournament + global elite** selection/crossover/mutation and vary **only how sensors are spawned**:

- **Oriented spawn** (`oriented=True`) — each sensor is aimed *outward and slightly down* (directional sensors use their node's surface-normal heading + downtilt, never pointing up; 360° sensors spawn near-upright). Avoids self-occlusion and lands in the scored region from gen 0.
- **Random spawn** (`oriented=False`) — today's fully random pitch/roll/yaw (the baseline).

Both build their initial population from the **same `SEED_INDIVIDUALS`** and run from the **same seed**, so the only controlled difference is the spawn orientation. Expect oriented to start markedly higher (verified ~14× higher mean fitness at gen 0) and ideally stay ≥ random.

> ⚠️ **Single seed (n=1).** Indicative only; repeat across several seeds for a defensible verdict.

In [ ]:
import config.seeding as seeding
import custom_toolbox.initialize.initialize as initialize
from utils.ga_run import run_evolution, TOURNAMENT
import utils.comparison as comparison

# Vary ONLY the spawn orientation; selection/crossover/mutation/seed/scale fixed.
# Each scheme builds its own initial population from the same seed individuals,
# using the matching toolbox generator (oriented vs fully random).
SPAWN_SCHEMES = {
    "Oriented spawn": toolbox.individual,          # oriented=True (default)
    "Random spawn": toolbox.individual_random,     # oriented=False
}

SPAWN_COMPARE_SEED = 23

spawn_results = {}
for label, generator in SPAWN_SCHEMES.items():
    init_pop = initialize.create_seeded_population(
        creator.Individual,  # type: ignore
        generator,           # the only thing that varies
        seeding.SEED_INDIVIDUALS,
        params.POPULATION_SIZE,
    )
    res = run_evolution(
        init_pop,
        toolbox,
        strategy=TOURNAMENT,  # original global-elite tournament, fixed
        label=label,
        seed=SPAWN_COMPARE_SEED,
    )
    spawn_results[res.label] = res

In [ ]:
# Overall best (solid) and average (dashed) fitness per spawn scheme.
# The gen-0 gap is the headline: oriented should start well above random.
comparison.compare_evolution(spawn_results)

In [ ]:
# Best layout per sensor count, per spawn scheme (table + grouped bars).
comparison.compare_best_per_length(spawn_results, evaluator=evaluator)

In [ ]:
# Per sensor count, both spawn schemes overlaid (best fitness over generations).
comparison.compare_per_length_evolution(spawn_results, metric="max")